# Download EMIMesh Data

This notebook downloads the EMIMesh repository and creates a dataset of meshed single cells and multi-cell cubes.

## Clone the modified EMIMesh repository
The ! runs a command the same way as writing it in the terminal

In [ ]:
# Clone the modified emimesh repository
REPO_URL = "https://github.com/SamiLaubo/emimesh.git"
DEST_DIR = "../emimesh"

!git clone $REPO_URL $DEST_DIR

Cloning into '../emimesh_repo'...
remote: Enumerating objects: 992, done.
remote: Counting objects: 100% (212/212), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 992 (delta 75), reused 129 (delta 52), pack-reused 780 (from 2)
Receiving objects: 100% (992/992), 1.38 MiB | 10.55 MiB/s, done.
Resolving deltas: 100% (505/505), done.


## Create environment

In [ ]:
# Install snakemake (using conda)
!conda create -c conda-forge -c bioconda -n snakemake snakemake snakemake-storage-plugin-http snakemake-executor-plugin-cluster-generic -y

Activate this environment in this notebook and use it when running the following codes.

## Create configuration files for different data

The dataset consists of the following configurations.

| **Name** | **# Cells** | **Resolution** | **# Samples** | **Cell Index** |
| --- | --- | --- | --- | --- |
| One neuron | 1 | All (1,2,3,4,5) | 10 | Sampled |
| One astrocyte | 1 | All (1,2,3,4,5) | 10 | Sampled |
| Microglia | 1 | All (1,2,3,4,5) | 5 | Sampled |
| Oligo | 1 | All (1,2,3,4,5) | 5 | Sampled |
| Pericyte | 1 | All (1,2,3,4,5) | 5 | Sampled |
| OPC | 1 | All (1,2,3,4,5) | 5 | Sampled |
| Two neurons | 2 | 1 | 5 | Sampled |
| Two astrocytes | 2 | 1 | 5 | Sampled |
| Small cube | 10 | 3 | 5 | Sampled |
| Small cube | 10 | 1 | 20 | Sampled |
| Medium cube | 30 | 1 | 20 | Sampled |
| Large cube | 100 | 1 | 20 | Sampled |

In [ ]:
# Configuration parameters for each cell type
cell_configs = {
    "neuron":       {"mips": [1, 2, 3, 4, 5], "samples": 10, "cell_max_size": 1000},
    "astrocyte":    {"mips": [1, 2, 3, 4, 5], "samples": 10, "cell_max_size": 1000},
    "microglia":    {"mips": [1, 2, 3, 4, 5], "samples": 5, "cell_max_size": 1000},
    "oligo":        {"mips": [1, 2, 3, 4, 5], "samples": 5, "cell_max_size": 1000},
    "pericyte":     {"mips": [1, 2, 3, 4, 5], "samples": 5, "cell_max_size": 1000},
    "OPC":          {"mips": [1, 2, 3, 4, 5], "samples": 5, "cell_max_size": 1000},
}

In [ ]:
import yaml
import os
import random

# Set seed for reproducibility
random.seed(42)

output_dir = "../emimesh/config_files/sscp_configs"
os.makedirs(output_dir, exist_ok=True)

# Template configuration based on neuron.yml
def generate_config(cell_type, mip, cell_idx, cell_max_size):
    config = {
        "name": f"{cell_type}_mip{mip}_smoothed",
        "raw": {
            "cloudpath": "precomputed://gs://iarpa_microns/minnie/minnie65/seg_m1300",
            "position": "0-0-0", # Not used in this context
            "mip": mip,
            "size": 5000,
            "cell_type": cell_type,
            "cell_idx": cell_idx,
            "cell_padding": 100,
            "cell_table_name": "aibs_metamodel_celltypes_v661",
            "cell_max_size": cell_max_size
        },
        "processing": {
            "dx": 20,
            "operation": [
                "smooth iterations=1 radius=2",
                "erode radius=1"
            ]
        },
        "meshing": {
            "envelopsize": 8
        }
    }
    return config

# Generate and save files
for cell_type, params in cell_configs.items():
    for sample_idx in range(params["samples"]):
        # Sample a random cell_idx for this specific cell
        cell_idx = random.randint(0, 100000)
        for mip in params["mips"]:
            # Include sample index in filename to avoid overwriting
            filename = f"{cell_type}_sample{sample_idx}_mip{mip}.yml"
            filepath = os.path.join(output_dir, filename)
            config_data = generate_config(cell_type, mip, cell_idx, params["cell_max_size"])

            with open(filepath, 'w') as f:
                yaml.dump(config_data, f, default_flow_style=False)
            
        print(f"Generated configs for {cell_type} sample {sample_idx}")

print("\nAll configuration files have been generated.")